# Phishing Email Detection with MLP: Overview & Workflow

This notebook provides a step-by-step guide to the complete hands-on process of phishing email detection using an MLP (Multi-Layer Perceptron) model.

## Workflow Summary

1. **Library Imports and Environment Setup**  
   Import all required libraries, set random seeds, and configure the environment for reproducibility.

2. **Data Loading**  
   Load the phishing email dataset into memory.

3. **Data Splitting**  
   Split the preprocessed dataset into training, validation, and test sets using stratified sampling to maintain class balance.

4. **Feature Extraction using TF-IDF**  
   Convert email texts into numerical feature vectors using TF-IDF vectorization.

5. **Feature Scaling and Tensor Conversion**  
   Scale features and convert them into PyTorch tensors for model input.

6. **MLP Model Definition**  
   Define the architecture of the Multi-Layer Perceptron model.

7. **Model Training**  
   Train the MLP model on the training set, validate on the validation set, and record loss curves.

8. **Loss Curve Visualization**  
   Visualize the training and validation loss curves to monitor learning and overfitting.

9. **Model Evaluation**  
   Evaluate the trained model on the test set using accuracy, F1-score, and confusion matrix.

---

# ?? ??? ?? MLP: ?? ? ?????

? ???? MLP(Multi-Layer Perceptron) ??? ??? ?? ??? ??? ?? ??? ???? ?????.

## ?? ?? ??

1. **????? ??? ? ?? ??**  
   ??? ?????? ?????, ?? ??? ???? ??? ???? ?????.

2. **??? ????**  
   ?? ??? ????? ???? ?????.

3. **??? ??**  
   ???? ???? ??, ??, ??? ??? ??? ???? ?? ?????.

4. **TF-IDF? ??? ?? ??**  
   ??? ???? TF-IDF ??? ???? ??? ???? ????.

5. **?? ???? ? ?? ??**  
   ??? ??????, PyTorch ??? ???? ?? ??? ???? ????.

6. **MLP ?? ??**  
   MLP ??? ??? ?????.

7. **?? ??**  
   ?? ???? ??? ?????, ?? ???? ??? ?????? ?? ??? ?????.

8. **?? ?? ???**  
   ?? ? ?? ?? ??? ????? ?? ? ??? ??? ?????.

9. **?? ??**  
   ??? ???? ??? ????, ???, F1-score, ???? ? ??? ??? ?????.

---


## 1. Library Imports and Environment Setup

This section imports all necessary Python libraries for the phishing email detection project.
- **pandas, numpy**: Used for data loading, cleaning, and numerical operations.
- **random, torch**: Used for deep learning model implementation and to ensure reproducibility by setting random seeds.
- **scikit-learn**: Used for data splitting, feature extraction, preprocessing, and evaluation metrics.
- **matplotlib**: Used for visualizing results.
- **warnings**: Used to suppress unnecessary warning messages for cleaner output.

All random seeds are fixed to ensure that experiments are fully reproducible.

---

## 1. ????? ??? ? ?? ??

? ????? ?? ??? ?? ????? ??? ??? ?????? ?? ??????.
- **pandas, numpy**: ??? ????, ??, ?? ??? ?????.
- **random, torch**: ??? ?? ??? ??? ???? ?? ????, ?? ?? ??? ?????.
- **scikit-learn**: ??? ??, ?? ??, ???, ?? ?? ??? ?????.
- **matplotlib**: ?? ???? ?????.
- **warnings**: ???? ?? ???? ???? ?? ??? ???? ????.

??? ??? ???? ???? ?? ?? ?? ??? ?????.


In [ ]:
import pandas as pd
import numpy as np
import random
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed(42)

## 2. Data Loading and Preprocessing

In this section, we load the phishing email dataset, perform basic data cleaning, encode the target labels, and preprocess the email text for further analysis. Specifically, we:
- Load the dataset from a CSV file.
- Remove unnecessary columns and handle missing or duplicate entries.
- Encode the 'Email Type' column into numerical labels (phishing: 0, safe: 1) for model compatibility.
- Apply a text preprocessing function to clean the email content by removing URLs, special characters, and extra spaces, and converting all text to lowercase.

This ensures that the data is clean, consistent, and ready for feature extraction and model training.

---

## 2. ??? ???? ? ???

? ????? ?? ??? ????? ????, ???? ??? ??, ?? ???, ??? ???? ?????. ?? ??? ??? ????.
- CSV ???? ????? ?????.
- ???? ?? ????, ??? ? ?? ???? ?????.
- 'Email Type' ??? ?? ??? ????? ??? ??(phishing: 0, safe: 1)? ?????.
- ??? ?? ????? URL, ????, ???? ??? ????, ???? ???? ??? ??? ?????.

?? ?? ???? ???? ??? ?? ????, ?? ?? ?? ? ?? ??? ??? ??? ?????.


In [ ]:
df = pd.read_csv('./Phishing_Email.csv')
if 'Unnamed: 0' in df.columns:
    df = df.drop('Unnamed: 0', axis=1)
df = df.dropna().drop_duplicates()

# Label encoding (phishing:0, safe:1)
le = LabelEncoder()
df['Email Type'] = le.fit_transform(df['Email Type'])

# Text preprocessing function
import re
def preprocess_text(text):
    text = re.sub(r'http\S+', '', text)  # Remove URLs
    text = re.sub(r'[^\w\s]', '', text)  # Remove special characters
    text = text.lower()                  # Convert to lowercase
    text = re.sub(r'\s+', ' ', text).strip()  # Remove extra spaces
    return text
df['Email Text'] = df['Email Text'].apply(preprocess_text)

print(df.head())

## 3. Data Splitting 

In this section, we split the preprocessed dataset into training, validation, and test sets. This is a crucial step to ensure that the model is trained on one subset of the data, validated on another for hyperparameter tuning and early stopping, and finally evaluated on a completely separate test set to assess its generalization performance. We use stratified sampling to maintain the original class distribution across all splits, which is especially important for imbalanced datasets like phishing detection.

- The dataset is first split into a training set (60%) and a temporary set (40%).
- The temporary set is then evenly divided into validation (20%) and test (20%) sets.
- The number of samples in each split is printed for verification.

---

## 3. ??? ??

? ????? ???? ????? ??, ??, ??? ??? ?????. ?? ??? ? ??? ???? ????, ?? ???? ??????? ?? ? ?? ??? ?? ??? ????, ????? ??? ??? ??? ???? ??? ??? ??? ? ??? ?? ??? ?????. ?? ???? ??? ???? ??? ? ?? ??????? ??? ??? ??? ?? ??? ???? ?? ?? ?????, ??? ???(stratified sampling)? ?????.

- ????? ?? ?? ??(60%)? ?? ??(40%)? ?????.
- ?? ??? ?? ?? ??(20%)? ??? ??(20%)? ???? ????.
- ? ??? ??? ?? ?? ???? ??? ???? ?????? ?????.


In [ ]:
texts = df['Email Text'].tolist()
labels = df['Email Type'].tolist()

x_train_text, x_temp_text, y_train, y_temp = train_test_split(
    texts, labels, test_size=0.4, random_state=42, stratify=labels
)
x_val_text, x_test_text, y_val, y_test = train_test_split(
    x_temp_text, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)
print(f'Train: {len(x_train_text)}, Val: {len(x_val_text)}, Test: {len(x_test_text)}')

## 4. Feature Extraction using TF-IDF

In this section, we convert the preprocessed email texts into numerical feature vectors using the TF-IDF (Term Frequency-Inverse Document Frequency) method. TF-IDF helps to quantify the importance of words in each email relative to the entire dataset, which is effective for text classification tasks like phishing detection.

- We use `TfidfVectorizer` from scikit-learn, removing English stop words and limiting the feature space to the top 10,000 terms.
- The vectorizer is first fitted on the entire email text corpus to learn the vocabulary.
- The training, validation, and test email texts are then transformed into TF-IDF feature arrays.
- The corresponding labels are also converted to NumPy arrays for compatibility with downstream processing.

---

## 4. TF-IDF? ??? ?? ??

? ????? ???? ??? ???? TF-IDF(?? ??-??? ??) ??? ???? ??? ?? ??? ?????. TF-IDF? ? ????? ??? ???? ?? ???? ???? ?????, ?? ??? ?? ??? ?? ??? ????? ?????.

- scikit-learn? `TfidfVectorizer`? ????, ?? ???(stop words)? ???? ?? 10,000?? ??? ???? ?????.
- ?? ??? ???? ?? ?????? ?? ??(fit)?? ??? ?????.
- ??, ??, ???? ??? ???? ?? TF-IDF ?? ??? ?????.
- ? ??? ??? ???? NumPy ??? ???? ?? ??? ???? ????.


In [ ]:
tfidf = TfidfVectorizer(stop_words='english', max_features=10000)
tfidf.fit(df['Email Text'])
x_train = tfidf.transform(x_train_text).toarray()
x_val = tfidf.transform(x_val_text).toarray()
x_test = tfidf.transform(x_test_text).toarray()
y_train = np.array(y_train)
y_val = np.array(y_val)
y_test = np.array(y_test)

## 5. Feature Scaling and Tensor Conversion

In this section, we standardize the TF-IDF feature vectors using `StandardScaler` from scikit-learn. Feature scaling ensures that each feature contributes equally to the model's learning process, which is especially important for neural networks. Since the TF-IDF vectors are sparse and already centered, we set `with_mean=False` to avoid centering and preserve sparsity.

After scaling, we convert the feature arrays and labels into PyTorch tensors, which are required for model training and evaluation.

---

## 5. ?? ???? ? ?? ??

? ????? scikit-learn? `StandardScaler`? ???? TF-IDF ?? ??? ??????. ?? ????? ? ??? ?? ??? ??? ????? ??, ?? ????? ?????. TF-IDF ??? ??(sparse)?? ?? ????? ????, `with_mean=False`? ???? ??? ?? ???? ?????.

???? ?, ?? ??? ???? PyTorch ??? ???? ?? ?? ? ??? ??? ? ??? ?????.


In [ ]:
scaler = StandardScaler(with_mean=False)
x_train_scaled = scaler.fit_transform(x_train)
x_val_scaled = scaler.transform(x_val)
x_test_scaled = scaler.transform(x_test)

x_train_tensor = torch.tensor(x_train_scaled, dtype=torch.float32)
x_val_tensor = torch.tensor(x_val_scaled, dtype=torch.float32)
x_test_tensor = torch.tensor(x_test_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32).unsqueeze(1)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

## 6. Define the MLP Model and Set Optimizer

In this section, we define a Multi-Layer Perceptron (MLP) model for phishing email classification.  
The MLP class inherits from PyTorch's `nn.Module` and dynamically sets the input dimension for the first linear layer.  
The model consists of two hidden layers (128 and 64 units) with ReLU activations and dropout (0.4) to prevent overfitting.  
The output layer uses a sigmoid activation to return probabilities for binary classification.

Next, we set up the binary cross-entropy loss function (`nn.BCELoss`) and the Adam optimizer for model training.  
The Adam optimizer is configured with a learning rate of 0.001 and L2 regularization (weight decay) of 5e-4 to improve generalization.

--- 

## 6. MLP ?? ?? ? ??? ?? ??

? ????? ?? ??? ??? ?? ?? ????(MLP) ??? ?????.  
PyTorch? `nn.Module`? ???? MLP ???? ????, ?? ??? ?? ? ?? ?? ??? ??? ???? ?????.  
??? ? ?? ???(128, 64 ??)? ReLU ??? ??, ????(0.4)?? ???? ???? ?????.  
???? ????? ??? ??? ???? ?? ?? ??? ?????.

??, ?? ?? ???? ?? ??(`nn.BCELoss`)? Adam ?????? ???? ?? ??? ??? ?? ??? ??? ????? ?????.  
Adam ?????? ??? 0.001? L2 ???(weight decay) 5e-4? ???? ??? ??? ????.




In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.model(x)

model = MLP(input_dim=x_train_tensor.shape[1])
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=5e-4)


## 7. Model Training and Early Stopping

In this section, we train the MLP model using the training dataset and validate its performance on the validation set.  
The training loop runs for a maximum of 30 epochs, with a batch size of 64.  
For each epoch, the training data is shuffled using a fixed random seed for reproducibility.  
The model is trained in mini-batches, and after each epoch, both training and validation losses, accuracy, and F1 scores are calculated.  

Early stopping is implemented to prevent overfitting:  
- If the validation loss does not improve for 3 consecutive epochs, training is stopped early.  
- The model parameters with the best validation loss are saved and restored after training.

---

## 7. ?? ?? ? Early Stopping ??

? ????? MLP ??? ?? ?????? ?????, ?? ????? ?? ??? ?????.  
?? ??? ?? 30 ??(epoch) ?? ????, ?? ??? 64? ?????.  
? ???? ??? ?? ??? ???? ?? ???? ???? ???? ?????.  
???? ??? ??? ?????, ? ??? ?? ??? ?? ? ?? ??, ???, F1 ??? ?????.  

Early stopping(?? ??) ??? ???? ???? ?????:  
- ?? ??? 3 ?? ???? ???? ??? ??? ??? ?????.  
- ?? ??? ?? ??? ??? ?? ????? ????, ??? ?? ? ?? ????? ?????.


In [ ]:
epochs = 30
batch_size = 64
train_losses, val_losses = [], []
train_accs, val_accs = [], []
train_f1s, val_f1s = [], []
best_val_loss = float('inf')
early_stop_count = 0
early_stop_patience = 3
import copy
for epoch in range(epochs):
    model.train()
    generator = torch.Generator()
    generator.manual_seed(42 + epoch)
    permutation = torch.randperm(x_train_tensor.size(0), generator=generator)
    for i in range(0, x_train_tensor.size(0), batch_size):
        indices = permutation[i:i+batch_size]
        batch_x = x_train_tensor[indices]
        batch_y = y_train_tensor[indices]
        optimizer.zero_grad()
        output = model(batch_x)
        loss = criterion(output, batch_y)
        loss.backward()
        optimizer.step()
    model.eval()
    with torch.no_grad():
        train_output = model(x_train_tensor)
        train_pred = (train_output > 0.5).int()
        train_loss = criterion(train_output, y_train_tensor).item()
        train_acc = accuracy_score(y_train_tensor, train_pred)
        train_f1 = f1_score(y_train_tensor, train_pred)
        val_output = model(x_val_tensor)
        val_pred = (val_output > 0.5).int()
        val_loss = criterion(val_output, y_val_tensor).item()
        val_acc = accuracy_score(y_val_tensor, val_pred)
        val_f1 = f1_score(y_val_tensor, val_pred)
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)
    train_f1s.append(train_f1)
    val_f1s.append(val_f1)
    print(f'Epoch {epoch+1}/{epochs} - Train Loss: {train_loss:.4f} - Val Loss: {val_loss:.4f} - Train Acc: {train_acc*100:.2f}% - Val Acc: {val_acc*100:.2f}% - Train F1: {train_f1*100:.2f}% - Val F1: {val_f1*100:.2f}%')
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        early_stop_count = 0
        best_model_state = copy.deepcopy(model.state_dict())
    else:
        early_stop_count += 1
        if early_stop_count >= early_stop_patience:
            print(f'Early stopping at epoch {epoch+1} (val loss ?? ??)')
            break
model.load_state_dict(best_model_state)

## 8. Loss Curve Visualization 

In this cell, we visualize the change in loss during model training.
The training and validation losses are plotted per epoch, allowing for intuitive monitoring of model learning and overfitting.

--- 

## 8. ?? ?? ??? 

? ???? ?? ?? ????? ??(Loss) ??? ??????.
?? ??? ?? ??? ???(epoch)?? ???? ????, ??? ?? ? ??? ??? ????? ??? ? ????.





In [ ]:
plt.figure(figsize=(10,4))
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss Curve')
plt.legend()
plt.show()

## 9. Model Evaluation on Test Set

In this section, we evaluate the performance of the trained MLP model on the test dataset. 
The model's predictions are generated using a threshold of 0.45, and the results are summarized with a classification report and a confusion matrix. 
This provides a comprehensive overview of the model's generalization ability and its effectiveness in distinguishing between phishing and safe emails.

---

## 9. ??? ?????? ?? ??

? ????? ??? MLP ??? ??? ?????? ?????.
??? ???? 0.45? ???(threshold)? ???? ????, ?? ???? ?? ??(confusion matrix)? ?? ??? ?????.
?? ?? ??? ?? ???? ?? ???? ??? ? ?????, ??? ?? ???? ?? ??? ??? ????? ??? ? ????.


In [ ]:
model = torch.load('mlp_best.pt')
model.eval()  

with torch.no_grad():
    test_output = model(x_test_tensor)
    test_pred = (test_output > 0.45).int()

print("\n[MLP] Test Results (threshold=0.45)\n")
print("Accuracy :", accuracy_score(y_test_tensor, test_pred))
print("F1 Score :", f1_score(y_test_tensor, test_pred))
print("\nClassification Report:\n", classification_report(y_test_tensor, test_pred, target_names=["phishing", "safe"]))

cm = confusion_matrix(y_test_tensor, test_pred)
ConfusionMatrixDisplay(cm, display_labels=["phishing", "safe"]).plot()
plt.title("Confusion Matrix - MLP (th=0.45)")
plt.show()
